## Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual [here](https://quantra.quantinsti.com/quantra-notebook).
2. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
3. Before running this notebook on your local PC:
   - You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on **"Run Codes Locally on Your Machine"** in the course.
   - You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or Python modules that might be required to run this notebook.

# Executing Large Trades with Iceberg Orders

Welcome! In our final notebook on order types, we address a unique challenge: how to execute very large orders without significantly impacting the market price. Placing a single, massive order can alert other market participants and cause the price to move against you before your full order is complete.

The solution is the **Iceberg order**, a specialized tool designed for discreetly executing large quantities.

### 1. Authentication

As always, we'll start by authenticating our Kite Connect client.

In [ ]:
from kiteconnect import KiteConnect
import logging

logging.basicConfig(level=logging.INFO)

# --- Credentials & Initialisation ---
api_key = "YOUR_API_KEY"
access_token = "YOUR_ACCESS_TOKEN"

try:
    kite = KiteConnect(api_key=api_key)
    kite.set_access_token(access_token)
    profile = kite.profile()
    print(f"Logged in as {profile['user_name']}")
    logging.info("Authentication successful!")
except Exception as e:
    logging.error(f"Authentication failed: {e}")

INFO:root:Authentication successful!


### 2. The Iceberg Order Concept

The name 'Iceberg' is the perfect analogy. You only show the small 'tip' of your order to the market, while the massive, hidden chunk remains below the surface.

It works by taking your large parent order and automatically breaking it into several smaller child orders, or **'legs'**. Only one leg is active on the exchange at any given time. Once that leg is fully executed, the system automatically sends the next leg, and so on, until the total quantity is filled.

To the market, this appears as a series of small, unrelated trades, which helps to minimize price impact and hide your true trading size.

### 3. Placing an Iceberg Order

The below code block demonstrates how to place an **Iceberg order** using the Kite Connect API.

First, we define the core parameters for our order:

* `symbol`: **ITC** - The trading symbol for the stock on the exchange (e.g., ITC on NSE).
* `price`: **435.0** - The limit price at which we want to buy the stock.
* `total_quantity`: **2000** - The total number of shares we intend to purchase.
* `iceberg_legs`: **4** - The number of smaller orders (legs) to split the total quantity into.
* `iceberg_quantity`: **500** - This is calculated automatically. It's the quantity for each individual leg, determined by the formula:
    $$
    iceberg\_quantity = \frac{total\_quantity}{iceberg\_legs} = \frac{2000}{4} = 500
    $$

In [ ]:
# --- Trading Symbol and Exchange Parameters ---
# Replace with your actual trading symbol and exchange
symbol = "ITC"
exchange = kite.EXCHANGE_NSE

# --- Iceberg Order Parameters ---
price = 435.0
total_quantity = 2000  # Total quantity for Iceberg order
iceberg_legs = 4  # Number of parts to break the order into
iceberg_quantity = total_quantity // iceberg_legs  # Each part's quantity

### Order Execution

The `try...except` block handles the execution of the order.

`kite.place_order()`: This function sends the order request to the exchange.
* `variety=kite.VARIETY_ICEBERG`: This parameter specifies that we are placing an Iceberg order.
* `transaction_type=kite.TRANSACTION_TYPE_BUY`: Sets the order as a "BUY" order.
* `order_type=kite.ORDER_TYPE_LIMIT`: A limit order will only execute at the specified price (`435.0`) or better.
* `product=kite.PRODUCT_CNC`: "Cash and Carry" (CNC) is for delivery-based trades where you hold the stock for more than one day.
* `quantity`: This is set to `total_quantity` (2000).
* `iceberg_legs`: This is set to `4`.
* `iceberg_quantity`: The quantity for the first disclosed leg (500).

If the order is placed successfully, the script logs the unique `order_id` for the parent Iceberg order. If there's an issue (e.g. invalid parameters), the `except` block catches the error and logs a descriptive message, preventing the program from crashing.

In [ ]:
try:
    order_id = kite.place_order(

        # Iceberg order type
        variety=kite.VARIETY_ICEBERG,

        exchange=exchange,
        tradingsymbol=symbol,

        # Transaction type
        transaction_type=kite.TRANSACTION_TYPE_BUY,

        order_type=kite.ORDER_TYPE_LIMIT,
        product=kite.PRODUCT_CNC,

        price=price,
        quantity=total_quantity,                       # Total quantity
        iceberg_legs=iceberg_legs,
        iceberg_quantity=iceberg_quantity,
    )
    logging.info(f"Iceberg order placed successfully. Parent Order ID: {order_id}")

except Exception as e:
    logging.error(f"Error placing Iceberg order: {e}")

INFO:root:Iceberg order placed successfully. Parent Order ID: 250717200172072


### 4. Conclusion

The Iceberg order is an essential tool for traders dealing with large volumes, allowing for more discreet execution with potentially less price impact.

You have now learned the entire toolkit available through the Kite Connect API, from basic Market and Limit orders to advanced, strategic types like GTT and Iceberg orders. This knowledge forms the core foundation for building any automated trading system.<br><br>